# 2 · Alloy blending (LP) · pair exercise

**Process industry.** A foundry must make a new alloy that is **30% lead, 30% zinc and 40% tin**. It can buy nine existing alloys (a–i), each with its own composition and price. Which mix of alloys makes the target composition at the lowest cost per kg?

This is Dantzig's classic blending problem (it is also in the GAMSPy examples).

Work in pairs. **Core** takes about 7 minutes. **Stretch** is for those who finish early.

In [1]:
import pandas as pd
import gamspy as gp

# percent of each element in each alloy, and price per kg
alloys = pd.DataFrame({
    "lead": [10, 10, 40, 60, 30, 30, 30, 50, 20],
    "zinc": [10, 30, 50, 30, 30, 40, 20, 40, 30],
    "tin":  [80, 60, 10, 10, 40, 30, 50, 10, 50],
    "cost": [4.1, 4.3, 5.8, 6.0, 7.6, 7.5, 7.3, 6.9, 7.3],
}, index=list("abcdefghi"))
target = {"lead": 30, "zinc": 30, "tin": 40}   # percent in the new alloy
alloys

,lead,zinc,tin,cost
a,10,10,80,4.1
b,10,30,60,4.3
c,40,50,10,5.8
d,60,30,10,6.0
e,30,30,40,7.6
f,30,40,30,7.5
g,30,20,50,7.3
h,50,40,10,6.9
i,20,30,50,7.3


In [2]:
m = gp.Container()

a = gp.Set(m, "a", description="alloys you can buy", records=alloys.index.tolist())
e = gp.Set(m, "e", description="elements", records=["lead", "zinc", "tin"])

comp = gp.Parameter(m, "comp", domain=[e, a], description="percent of element e in alloy a",
                    records=alloys[["lead", "zinc", "tin"]].T.stack().reset_index())
price = gp.Parameter(m, "price", domain=a, description="price per kg",
                     records=alloys["cost"].reset_index())
req = gp.Parameter(m, "req", domain=e, description="required percent in the new alloy",
                   records=list(target.items()))

# w[a] = fraction of alloy a in 1 kg of the new alloy
w = gp.Variable(m, "w", domain=a, type="positive")

## Core: write the model

You need two kinds of constraints and an objective:

1. **Fractions add up to one:** the fractions of all alloys sum to 1 kg.
2. **Composition:** for every element e, the blended percent equals the target: $\sum_a \text{comp}_{e,a}\, w_a = \text{req}_e$.
3. **Objective:** minimize the cost of 1 kg, $\sum_a \text{price}_a\, w_a$.

Replace each `...` below. Hints: `gp.Sum(a, ...)`, `==` for equality.

In [3]:
one = gp.Equation(m, "one", description="fractions sum to 1")
composition = gp.Equation(m, "composition", domain=e)

one[...] = gp.Sum(a, w[a]) == 1
composition[e] = gp.Sum(a, comp[e, a] * w[a]) == req[e]

blend = gp.Model(m, "blend", equations=[one, composition],
                 problem="LP", sense="min", objective=gp.Sum(a, price[a] * w[a]))
blend.solve()
print(blend.status, round(blend.objective_value, 3))
w.records[w.records.level > 1e-6][["a", "level"]]

ModelStatus.OptimalGlobal 4.98


,a,level
1,b,0.6
3,d,0.4


In [4]:
# Check your answer
assert abs(blend.objective_value - 4.98) < 1e-6, "Not quite: the cheapest blend costs 4.98 per kg"
print("Correct! 60% alloy b + 40% alloy d gives exactly 30/30/40 for 4.98 per kg.")

Correct! 60% alloy b + 40% alloy d gives exactly 30/30/40 for 4.98 per kg.


**Discuss:** why does the cheapest blend use only two alloys out of nine? (Hint: count the constraints.)

**Core, part 2:** alloy b's supplier raises its price from 4.3 to 5.0. Change the data, re-solve, and see which alloys the model picks now.

In [5]:
price.setRecords(alloys["cost"].replace({4.3: 5.0}).reset_index())
blend.solve()
print(round(blend.objective_value, 3))
display(w.records[w.records.level > 1e-6][["a", "level"]])
price.setRecords(alloys["cost"].reset_index())   # put the data back

5.1


,a,level
0,a,0.428571
2,c,0.428571
3,d,0.142857


## Stretch: when streams mix first (the pooling problem)

In a real plant, some streams are mixed in a **tank** before they reach the products. Then the tank's composition is itself unknown, and *composition × flow* is a product of two variables. The model becomes **nonlinear and nonconvex**.

This is Haverly's pooling problem, a classic from oil refining:

- Crudes **A** (3% sulfur, cost 6) and **B** (1% sulfur, cost 16) go into a pool.
- Crude **C** (2% sulfur, cost 10) bypasses the pool.
- Product **X**: at most 2.5% sulfur, sells for 9, demand at most 100.
- Product **Y**: at most 1.5% sulfur, sells for 15, demand at most 200.

`q` is the sulfur percent in the pool. It multiplies flows, so the problem is an NLP. Run it from two different starting points.

In [6]:
def haverly(q_start, flow_start):
    m = gp.Container()
    A, B, PX, PY, CX, CY = [gp.Variable(m, n, type="positive") for n in ["A", "B", "PX", "PY", "CX", "CY"]]
    q = gp.Variable(m, "q", description="sulfur percent in the pool")
    q.lo[...] = 1
    q.up[...] = 3
    q.l[...] = q_start                      # starting point for the solver
    for v in [A, B, PX, PY, CX, CY]:
        v.l[...] = flow_start

    pool_flow = gp.Equation(m, "pool_flow")
    pool_sulfur = gp.Equation(m, "pool_sulfur")
    spec_x = gp.Equation(m, "spec_x")
    spec_y = gp.Equation(m, "spec_y")
    dem_x = gp.Equation(m, "dem_x")
    dem_y = gp.Equation(m, "dem_y")

    pool_flow[...]   = A + B == PX + PY
    pool_sulfur[...] = 3 * A + 1 * B == q * (PX + PY)        # bilinear: q times a flow
    spec_x[...]      = q * PX + 2 * CX <= 2.5 * (PX + CX)
    spec_y[...]      = q * PY + 2 * CY <= 1.5 * (PY + CY)
    dem_x[...]       = PX + CX <= 100
    dem_y[...]       = PY + CY <= 200

    profit = 9 * (PX + CX) + 15 * (PY + CY) - 6 * A - 16 * B - 10 * (CX + CY)
    model = gp.Model(m, "haverly", equations=m.getEquations(), problem="NLP",
                     sense="max", objective=profit)
    model.solve(solver="CONOPT")
    return model.objective_value, q.toValue(), model.status

for q_start, flow_start in [(3, 0), (1, 50)]:
    profit, q_final, status = haverly(q_start, flow_start)
    print(f"start q={q_start}, flows={flow_start}:  profit = {profit:6.1f}   pool sulfur = {q_final:.2f}%   ({status.name})")

start q=3, flows=0:  profit =  100.0   pool sulfur = 3.00%   (OptimalLocal)
start q=1, flows=50:  profit =  400.0   pool sulfur = 1.00%   (OptimalLocal)


**Discuss:** both answers are reported as *optimal*, but one earns four times more. CONOPT is a **local** solver: it finds the best point *near where it started*. For nonconvex models you need several starting points, or a **global** solver (such as BARON or SCIP, installable with `gamspy install solver`) that can prove which answer is best. We come back to this in Act 3.